# Чистовик: константный прогноз

Заготовка, которая уже проходит весь путь: читает `data/`, считает CV по фолдам, пишет `submission.csv` и `cv.json` и печатает `cv.json` последней строкой. Замените прогноз и разбиение на свои.

In [1]:
import json, sys
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, GroupKFold

sys.path.insert(0, "data")
from metric_cv import quantile_quality, cv_json

tr = pd.read_csv("data/train.csv")
te = pd.read_csv("data/test.csv")
sample = pd.read_csv("data/sample_submission.csv")
y = tr["daily_orders"].to_numpy(float)
QUANTILES = (0.1, 0.5, 0.9)


def predict(y_train, n):
    """Константный прогноз: квантили спроса по всему train. Замените на модель."""
    q = np.quantile(y_train, QUANTILES)
    return np.tile(q, (n, 1))


# CV: четыре случайных фолда — замените разбиением, которое похоже на тест
gkf = GroupKFold(n_splits=4)
splits = list(gkf.split(tr, y, groups=tr['city']))
oof = np.zeros((len(tr), 3))
for a, b in splits:
    oof[b] = predict(y[a], len(b))
folds = [100 * quantile_quality(y[b], oof[b, 0], oof[b, 1], oof[b, 2], tr["city"].iloc[b]) for _, b in splits]

# посылка: колонки и строки как в sample_submission.csv, 0 <= p10 <= p50 <= p90 <= 5000
test = np.round(predict(y, len(te)), 1)
sub = pd.DataFrame({"id": te["id"],
                    "predicted_daily_orders_p10": test[:, 0],
                    "predicted_daily_orders_p50": test[:, 1],
                    "predicted_daily_orders_p90": test[:, 2]})
assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample)
assert set(sub["id"]) == set(sample["id"]) and (sub.iloc[:, 1:].to_numpy() >= 0).all()
sub.to_csv("submission.csv", index=False)

cv = cv_json("constant", [f / 100 for f in folds])   # пишет cv.json
print(json.dumps(cv))                                  # последняя строка — для проверки воспроизводимости


{"name": "constant", "metric": "quantile_quality_x100", "folds": [0.0, 0.0, 0.0, 0.0], "cv": 0.0, "std": 0.0}
